In [ ]:
import pandas as pd
import os
import re
import ast
import time
import fitz  # PyMuPDF
import getpass
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

print("1. Setting up API & LLM...")
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key (gsk_...): ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("\n2. Processing 1 PDF (Volume 1) with BASELINE CHUNKING...")
pdf_path = "../data/raw/university-physics-volume-1.pdf"
all_chunks = []

def baseline_chunker(text, max_chars=1000):
    paragraphs = text.split('\n\n')
    chunks, current = [], ""
    for p in paragraphs:
        if len(current) + len(p) < max_chars:
            current += p + "\n"
        else:
            if current.strip(): chunks.append(current.strip())
            current = p + "\n"
    if current.strip(): chunks.append(current.strip())
    return chunks

if not os.path.exists(pdf_path):
    print(f"Error: {pdf_path} not found!")
else:
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num in range(min(300, len(doc))):
        full_text += doc.load_page(page_num).get_text("text") + "\n\n"
    all_chunks = baseline_chunker(full_text)
    print(f"-> Extracted {len(all_chunks)} Baseline Chunks from Volume 1.")

print("\n3. Building Vector Database (Baseline)...")
vector_db = Chroma.from_texts(texts=all_chunks, embedding=embedding_model, collection_name="baseline_small_db")

print("\n4. Hard-filtering Exactly 7 Mechanics Questions (Volume 1)...")
csv_path = "../data/benchmark/physics_evaluation_benchmark.csv"
df = pd.read_csv(csv_path)

mechanics_indices = [11, 17, 26, 31, 32, 43, 46]
df_vol1 = df.iloc[mechanics_indices].copy()
print(f"-> Selected {len(df_vol1)} absolute mechanics questions.")

print("\n5. Running Evaluation Loop...")
prompt_template = PromptTemplate(
    input_variables=["context", "question", "choices"],
    template="""You are an expert physics professor answering a multiple-choice exam.
Read the context, question, and choices carefully.
You MUST provide your final answer wrapped in XML tags like this: <answer>INDEX</answer>.
If the answer is not clearly in the context, you MUST guess the most logical index based on physics principles.
Only output the XML tag, no other text!

CONTEXT:
{context}

QUESTION: 
{question}

CHOICES:
{choices}

Provide your answer:"""
)

def extract_choices(choices_str):
    matches = re.findall(r"'([^']*)'", str(choices_str))
    if len(matches) != 4: matches = re.findall(r'"([^"]*)"', str(choices_str))
    return matches

evaluation_results = []

for index, row in df_vol1.iterrows():
    q_text = row['question']
    expected_ans = row['answer']
    choices_list = extract_choices(row['choices'])
    choices_text = "\n".join([f"Index {i}: {choice}" for i, choice in enumerate(choices_list)])

    retrieved_docs = vector_db.similarity_search(q_text, k=3)
    c_text = "\n\n---\n\n".join([doc.page_content for doc in retrieved_docs])
    
    final_prompt = prompt_template.format(context=c_text, question=q_text, choices=choices_text)
    
    for attempt in range(5):
        try:
            response_text = llm.invoke(final_prompt).content
            time.sleep(2)
            break
        except Exception as e:
            if "429" in str(e) or "rate_limit" in str(e).lower():
                time.sleep(20)
            else:
                raise e
    
    match = re.search(r"<answer>\s*(\d+)\s*</answer>", response_text, re.IGNORECASE)
    if match:
        pred_ans = match.group(1)
    else:
        digits = re.findall(r"\d", response_text)
        pred_ans = digits[0] if digits else "0"
        
    print(f"Q: {q_text[:50]}...")
    print(f"Expected: {expected_ans} | Predicted: {pred_ans}")
    print("-" * 30)
    
    evaluation_results.append({
        "Question": q_text,
        "Choices": str(choices_list),
        "Expected_Answer": expected_ans,
        "Predicted_Answer": pred_ans,
        "Retrieved_Context": c_text
    })

print("\n6. Saving Baseline Small Scale Results...")
results_df = pd.DataFrame(evaluation_results)
output_path = "../data/benchmark/eval_01_baseline_small.csv"
results_df.to_csv(output_path, index=False)
print(f"DONE! Results saved to '{output_path}'")

1. Setting up API & LLM...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


2. Processing 1 PDF (Volume 1) with BASELINE CHUNKING...
-> Extracted 297 Baseline Chunks from Volume 1.

3. Building Vector Database (Baseline)...

4. Hard-filtering Exactly 7 Mechanics Questions (Volume 1)...
-> Selected 7 absolute mechanics questions.

5. Running Evaluation Loop...
Q: The coefficient of static friction between a small...
Expected: 3 | Predicted: 2
------------------------------
Q: One end of a horizontal, massless spring is attach...
Expected: 0 | Predicted: 1
------------------------------
Q: An astronomer observes a very small moon orbiting ...
Expected: 0 | Predicted: 2
------------------------------
Q: If the Sun were suddenly replaced by a black hole ...
Expected: 3 | Predicted: 2
------------------------------
Q: At 20°C, a pipe open at both ends resonates at a f...
Expected: 1 | Predicted: 1
------------------------------
Q: A uniform solid disk starts from rest and rolls do...
Expected: 1 | Predicted: 2
------------------------------
Q: The driver of a poli